# Day 12: Model Evaluation Metrics

## Setup
Before you can evaluate any metric, you need a trained model producing predictions to measure — this cell reuses the familiar Iris classification setup (load data, split into train/test, fit a logistic regression) purely as a foundation for the metric examples that follow. `test_size=0.3` reserves 30% of the data for testing (a bit more than usual, giving a slightly larger sample to compute metrics on), and `y_pred` stores the model's predicted class for every test row, which we'll compare against the true labels `y_test` throughout this notebook. There's no printed output here, but this cell produces `y_test` and `y_pred` — the two ingredients every evaluation metric below needs.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=200).fit(X_train, y_train)  # train the model
y_pred = model.predict(X_test)  # predictions we'll evaluate against y_test below

## Confusion Matrix
A confusion matrix is one of the most useful diagnostic tools for classification — instead of collapsing performance into a single number, it shows exactly which classes the model confuses with which, which is essential when some mistakes matter more than others (e.g. missing a disease diagnosis vs. a false alarm). `confusion_matrix(y_test, y_pred)` builds a grid where row `i` is the true class and column `j` is the predicted class, so each cell counts how many samples of true class `i` were predicted as class `j`. In the output, all the non-zero values sit on the diagonal (19, 13, 13 correct predictions for classes 0, 1, 2), and every off-diagonal cell is 0, meaning the model made zero misclassifications on this test set.

In [ ]:
from sklearn.metrics import confusion_matrix
# rows = true class, columns = predicted class; diagonal = correct predictions
print(confusion_matrix(y_test, y_pred))

## Accuracy, Precision, Recall, F1
Accuracy alone can be misleading, especially with imbalanced classes (a model that always predicts the majority class can still look "accurate"), so it's important to know a fuller set of metrics and what each one actually penalizes. Accuracy is the overall fraction of correct predictions; precision asks "of everything predicted as class X, how much was actually class X" (penalizes false positives); recall asks "of everything that's actually class X, how much did we catch" (penalizes false negatives); F1 is the harmonic mean of precision and recall, balancing both. `average='macro'` computes each metric separately per class and then averages them equally, treating every class as equally important regardless of how many samples it has; the output shows all four metrics at a perfect 1.0, consistent with the confusion matrix showing zero misclassifications.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print('Accuracy:', accuracy_score(y_test, y_pred))   # overall fraction of correct predictions
# average='macro' computes the metric per class, then averages the classes equally
print('Precision:', precision_score(y_test, y_pred, average='macro'))
print('Recall:', recall_score(y_test, y_pred, average='macro'))
print('F1 Score:', f1_score(y_test, y_pred, average='macro'))

## Classification Report
Rather than computing accuracy, precision, recall, and F1 separately, `classification_report` is a convenient shortcut that prints all of them at once, broken down per class — this is usually the first thing you'd print when checking how a classifier performs, since it gives you the complete picture in one readable table. It also reports `support` (how many true samples exist for each class in the test set) plus macro and weighted averages, where the weighted average accounts for class imbalance by weighting each class's contribution by its support. In the output, every class shows perfect precision/recall/f1-score of 1.00, and the `support` column (19, 13, 13) confirms how many test samples belonged to each class, matching the confusion matrix.

In [ ]:
from sklearn.metrics import classification_report
# One table with precision, recall, f1-score, and support (sample count) for every class
print(classification_report(y_test, y_pred))

## Cross-Validation
A single train/test split can give a misleadingly optimistic or pessimistic score just by chance, depending on which rows happened to land in the test set — cross-validation addresses this by repeating the train/test process multiple times on different splits and averaging the results, giving a more reliable estimate of how the model will perform on new data. `cross_val_score(model, X, y, cv=5)` performs 5-fold cross-validation: it splits the full dataset into 5 equal parts ("folds"), and for each of the 5 rounds trains on 4 folds while testing on the remaining fold, returning one accuracy score per round. In the output, `CV Scores` shows the 5 individual accuracy results (ranging from ~0.93 to 1.0), and `Average` (~0.97) is generally a more trustworthy performance estimate than any single train/test split score.

In [ ]:
from sklearn.model_selection import cross_val_score

# cv=5: run 5 separate train/test rounds on different folds of the full dataset
scores = cross_val_score(LogisticRegression(max_iter=200), X, y, cv=5)
print('CV Scores:', scores)     # accuracy from each of the 5 rounds
print('Average:', scores.mean())  # more reliable overall performance estimate